# SQL Important Questions

#  1️⃣ KPI & EXECUTIVE SUMMARY (Foundation Layer)

## 🎯 Objective:

Understand overall business health.

## 🔍 Must Calculate:

* Total Revenue
* Total Profit
* Profit Margin %
* Total Orders
* Total Customers
* Average Order Value (AOV)

## 🧠 SQL Concepts:

* `SUM()`
* `COUNT()`
* `COUNT(DISTINCT)`
* Arithmetic calculations

## 💼 Business Question:

> “Is the business profitable and growing in scale?”

## 🎤 Interview Line:

> “I started by building executive KPIs using SQL aggregation functions to quantify total revenue, profitability, order volume, and customer base.”

---

1️⃣ KPI Summary

```
SELECT
    ROUND(SUM(revenue)::NUMERIC, 2) AS total_revenue,
    ROUND(SUM(profit)::NUMERIC, 2) AS total_profit,
    ROUND((SUM(profit) / SUM(revenue) * 100)::NUMERIC, 2) AS profit_margin_pct,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_unique_id) AS total_customers,
    ROUND(AVG(revenue)::NUMERIC, 2) AS avg_order_value
FROM olist;
```

# 🔵 2️⃣ TIME-SERIES ANALYSIS (Growth Intelligence)


## 🔍 Must Perform:

* Monthly Revenue
* Monthly Profit
* Monthly Orders
* Month-over-Month (MoM) Growth %

## 🧠 SQL Concepts:

* `DATE_TRUNC()` / `EXTRACT(MONTH)`
* `GROUP BY`
* `LAG()` window function
* Percentage calculations

## ⭐ Advanced:

Use `LAG()` to calculate MoM growth:

Revenue Growth % =
[
(current_month - previous_month) / previous_month
]

## 💼 Business Question:

> “Is revenue accelerating or slowing down?”

## 🎤 Interview Line:

> “I used window functions like LAG to compute month-over-month revenue growth, enabling trend diagnostics beyond simple aggregation.”

This sounds strong.

---

2️⃣ Monthly Trend + MoM Growth

```
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', order_purchase_timestamp) AS purchase_month,
        ROUND(SUM(revenue)::NUMERIC, 2) AS total_revenue,
        ROUND(SUM(profit)::NUMERIC, 2) AS total_profit,
        COUNT(DISTINCT order_id) AS total_orders
    FROM olist
    GROUP BY 1
)
SELECT
    purchase_month,
    total_revenue,
    total_profit,
    total_orders,
    LAG(total_revenue) OVER (ORDER BY purchase_month) AS prev_revenue,
    ROUND((
        (total_revenue - LAG(total_revenue) OVER (ORDER BY purchase_month))
        / NULLIF(LAG(total_revenue) OVER (ORDER BY purchase_month), 0)
        * 100
    )::NUMERIC, 2) AS mom_growth_pct
FROM monthly
ORDER BY 1;
```

# 🔵 3️⃣ CATEGORY PERFORMANCE ANALYSIS

## 🔍 Must Calculate:

* Revenue by category
* Profit by category
* Profit margin %
* Top 10 categories

## 🧠 SQL Concepts:

* `GROUP BY`
* `ORDER BY DESC`
* `LIMIT`
* `CASE` for margin classification
* `HAVING`

## ⭐ Advanced:

Classify categories:

```sql
CASE 
  WHEN margin > 30 THEN 'High Margin'
  WHEN margin BETWEEN 15 AND 30 THEN 'Medium Margin'
  ELSE 'Low Margin'
END
```

## 💼 Business Question:

> “Which product categories drive revenue vs profit?”

## 🎤 Interview Line:

> “I segmented product categories by profit margin to identify high-volume low-margin versus high-margin strategic products.”

---


3️⃣ Category Performance

```
SELECT
    product_category_name,
    ROUND(SUM(revenue)::NUMERIC, 2) AS total_revenue,
    ROUND(SUM(profit)::NUMERIC, 2) AS total_profit,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND((SUM(profit) / SUM(revenue) * 100)::NUMERIC, 2) AS profit_margin_pct,
    CASE
        WHEN SUM(profit) / SUM(revenue) * 100 > 30 THEN 'High Margin'
        WHEN SUM(profit) / SUM(revenue) * 100 BETWEEN 15 AND 30 THEN 'Medium Margin'
        ELSE 'Low Margin'
    END AS margin_class
FROM olist
GROUP BY 1
ORDER BY total_revenue DESC
LIMIT 10;
```

# 🔵 4️⃣ STATE-WISE / GEOGRAPHIC PERFORMANCE

## 🔍 Must Calculate:

* Revenue by state
* Profit margin by state
* Orders by state
* Top performing states

## 🧠 SQL Concepts:

* `GROUP BY`
* `ROUND()`
* `WHERE`
* `ORDER BY`
* `HAVING`

## 💼 Business Question:

> “Which regions are profitable and scalable?”

## 🎤 Interview Line:

> “I analyzed geographic profitability to identify states with strong revenue but weak margins, highlighting operational inefficiencies.”

---


4️⃣ State Performance

```
SELECT
    customer_state,
    ROUND(SUM(revenue)::NUMERIC, 2) AS total_revenue,
    ROUND(SUM(profit)::NUMERIC, 2) AS total_profit,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_unique_id) AS total_customers,
    ROUND((SUM(profit) / SUM(revenue) * 100)::NUMERIC, 2) AS profit_margin_pct,
    ROUND(AVG(revenue)::NUMERIC, 2) AS avg_order_value
FROM olist
GROUP BY 1
ORDER BY total_revenue DESC;
```

# 🔵 5️⃣ CUSTOMER INTELLIGENCE (Most Important Section)

## 🔹 A. Customer Revenue Ranking

### 🔍 Perform:

* Total revenue per customer
* Rank customers

### 🧠 SQL Concepts:

* `GROUP BY`
* `SUM()`
* `RANK()` or `DENSE_RANK() OVER()`

### 💼 Business Question:

> “Who are the top revenue-generating customers?”

### 🎤 Interview Line:

> “I ranked customers using window functions to identify high-value accounts and revenue concentration.”

---

5️⃣ A — Customer Revenue Ranking

```
SELECT
    customer_unique_id,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(revenue)::NUMERIC, 2) AS total_revenue,
    ROUND(SUM(profit)::NUMERIC, 2) AS total_profit,
    RANK() OVER (ORDER BY SUM(revenue) DESC) AS revenue_rank
FROM olist
GROUP BY 1
ORDER BY total_revenue DESC
LIMIT 20;
```

## 🔹 B. Pareto Analysis (80/20 Rule)

### 🔍 Perform:

* Revenue per customer
* Cumulative revenue
* Cumulative revenue %

### 🧠 SQL Concepts:

* `SUM() OVER(ORDER BY revenue DESC)`
* Running totals
* Window functions

### 💼 Business Question:

> “Is revenue concentrated among a small percentage of customers?”

### 🎤 Interview Line:

> “I implemented cumulative revenue analysis using window functions to validate the Pareto principle and assess revenue dependency risk.”

Very strong.

---

5️⃣ B — Pareto Analysis

```
WITH customer_revenue AS (
    SELECT
        customer_unique_id,
        ROUND(SUM(revenue)::NUMERIC, 2) AS total_revenue
    FROM olist
    GROUP BY 1
),
cumulative AS (
    SELECT
        customer_unique_id,
        total_revenue,
        SUM(total_revenue) OVER (ORDER BY total_revenue DESC) AS cumulative_revenue,
        SUM(total_revenue) OVER () AS grand_total
    FROM customer_revenue
)
SELECT
    customer_unique_id,
    total_revenue,
    ROUND((cumulative_revenue / grand_total * 100)::NUMERIC, 2) AS cumulative_pct
FROM cumulative
ORDER BY total_revenue DESC
LIMIT 20;
```

## 🔹 C. RFM Base Metrics

### 🔍 Calculate:

* Recency → `DATEDIFF(current_date, MAX(order_date))`
* Frequency → `COUNT(DISTINCT order_id)`
* Monetary → `SUM(revenue)`

### 🧠 SQL Concepts:

* `MAX()`
* `DATEDIFF()`
* `GROUP BY`
* `CASE` (optional scoring)

### 💼 Business Question:

> “Who are loyal vs churn-risk customers?”

### 🎤 Interview Line:

> “I built the RFM base table in SQL, calculating recency, frequency, and monetary values to support segmentation modeling.”

---

5️⃣ C — RFM Base Metrics

```
SELECT
    customer_unique_id,
    MAX(order_purchase_timestamp)::DATE AS last_purchase_date,
    CURRENT_DATE - MAX(order_purchase_timestamp)::DATE AS recency_days,
    COUNT(DISTINCT order_id) AS frequency,
    ROUND(SUM(revenue)::NUMERIC, 2) AS monetary
FROM olist
GROUP BY 1
ORDER BY monetary DESC;
```

## 🔹 D. Cohort Analysis (Advanced Level)

5️⃣ D — Cohort Analysis

```
WITH cohort AS (
    SELECT
        customer_unique_id,
        DATE_TRUNC('month', MIN(order_purchase_timestamp)) AS cohort_month
    FROM olist
    GROUP BY 1
),
orders AS (
    SELECT
        o.customer_unique_id,
        DATE_TRUNC('month', o.order_purchase_timestamp) AS order_month,
        c.cohort_month,
        EXTRACT(YEAR FROM AGE(
            DATE_TRUNC('month', o.order_purchase_timestamp), c.cohort_month
        )) * 12 +
        EXTRACT(MONTH FROM AGE(
            DATE_TRUNC('month', o.order_purchase_timestamp), c.cohort_month
        )) AS cohort_index
    FROM olist o
    JOIN cohort c ON o.customer_unique_id = c.customer_unique_id
)
SELECT
    cohort_month,
    cohort_index,
    COUNT(DISTINCT customer_unique_id) AS customers
FROM orders
GROUP BY 1, 2
ORDER BY 1, 2;
```


# 🔵 6️⃣ PROFITABILITY DEEP DIVE


### 🔍 Perform:

* First purchase month per customer
* Assign cohort month
* Track repeat purchases month-wise

### 🧠 SQL Concepts:

* `MIN(order_date)`
* `DATE_TRUNC()`
* Self join on customer_id
* Month difference calculation

### 💼 Business Question:

> “How well do customers retain over time?”

### 🎤 Interview Line:

> “I constructed cohort retention tables using first purchase month logic to analyze behavioral retention patterns.”

---

## 🔹 A. Negative Profit Orders


```sql
WHERE profit < 0
```



### Business Question:

> “Are we losing money on certain transactions?”

---

## 🔹 B. High Freight Ratio

Calculate:

```
freight_ratio = freight_value / revenue
```

### Business Question:

> “Is logistics cost hurting margins?”

---

6️⃣ A — High Freight Ratio

```
```
SELECT
    product_category_name,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(AVG(freight_value / NULLIF(revenue, 0))::NUMERIC, 4) AS avg_freight_ratio,
    ROUND(SUM(revenue)::NUMERIC, 2) AS total_revenue,
    ROUND(SUM(profit)::NUMERIC, 2) AS total_profit
FROM olist
WHERE freight_value / NULLIF(revenue, 0) > 0.4
GROUP BY 1
ORDER BY total_orders DESC
LIMIT 10;
```
```

## 🔹 C. Low Margin States or Categories




Use:

```
HAVING SUM(profit)/SUM(revenue) < 0.10
```

### Business Question:

> “Where is cost leakage happening?”

---

6️⃣ B — Low Margin States

```
SELECT
    customer_state,
    ROUND(SUM(revenue)::NUMERIC, 2) AS total_revenue,
    ROUND(SUM(profit)::NUMERIC, 2) AS total_profit,
    ROUND((SUM(profit) / SUM(revenue) * 100)::NUMERIC, 2) AS profit_margin_pct
FROM olist
GROUP BY 1
HAVING SUM(profit) / SUM(revenue) < 0.10
ORDER BY profit_margin_pct ASC;
```

# 🔵 7️⃣ ADVANCED SQL EDGE (Optional but Powerful)

---
If you want to look senior-level:

* Running total revenue
* Year-over-Year growth
* Top N per category using `ROW_NUMBER()`
* Customer segmentation using `CASE`
* Percentile analysis using `NTILE()`

---

7️⃣ ADVANCED SQL

```
SELECT
    DATE_TRUNC('month', order_purchase_timestamp) AS purchase_month,
    ROUND(SUM(revenue)::NUMERIC, 2) AS monthly_revenue,
    ROUND(SUM(SUM(revenue)) OVER (ORDER BY DATE_TRUNC('month', order_purchase_timestamp))::NUMERIC, 2) AS running_total
FROM olist
GROUP BY 1
ORDER BY 1;
```

Top N Per Category using ROW_NUMBER:

In [ ]:
```
WITH ranked AS (
    SELECT
        product_category_name,
        customer_unique_id,
        ROUND(SUM(revenue)::NUMERIC, 2) AS total_revenue,
        ROW_NUMBER() OVER (PARTITION BY product_category_name ORDER BY SUM(revenue) DESC) AS rn
    FROM olist
    GROUP BY 1, 2
)
SELECT * FROM ranked WHERE rn <= 3
ORDER BY product_category_name, rn;
```

Customer Segmentation using CASE:

```
WITH rfm AS (
    SELECT
        customer_unique_id,
        CURRENT_DATE - MAX(order_purchase_timestamp)::DATE AS recency,
        COUNT(DISTINCT order_id) AS frequency,
        ROUND(SUM(revenue)::NUMERIC, 2) AS monetary
    FROM olist
    GROUP BY 1
)
SELECT
    customer_unique_id,
    recency,
    frequency,
    monetary,
    CASE
        WHEN recency <= 90  AND frequency >= 3 AND monetary >= 500 THEN 'Champions'
        WHEN recency <= 180 AND frequency >= 2                     THEN 'Loyal'
        WHEN recency <= 90  AND frequency = 1                      THEN 'New Customer'
        WHEN recency > 180  AND frequency >= 2                     THEN 'At Risk'
        ELSE 'Lost'
    END AS segment
FROM rfm
ORDER BY monetary DESC;
```

In [ ]:
NTILE Percentile Analysis:

In [ ]:
```
SELECT
    customer_unique_id,
    ROUND(SUM(revenue)::NUMERIC, 2) AS total_revenue,
    NTILE(4) OVER (ORDER BY SUM(revenue)) AS revenue_quartile,
    NTILE(10) OVER (ORDER BY SUM(revenue)) AS revenue_decile
FROM olist
GROUP BY 1
ORDER BY total_revenue DESC;
```

In [ ]:
Year over Year Growth:

```
WITH yearly AS (
    SELECT
        EXTRACT(YEAR FROM order_purchase_timestamp) AS yr,
        ROUND(SUM(revenue)::NUMERIC, 2) AS total_revenue
    FROM olist
    GROUP BY 1
)
SELECT
    yr,
    total_revenue,
    LAG(total_revenue) OVER (ORDER BY yr) AS prev_year_revenue,
    ROUND((
        (total_revenue - LAG(total_revenue) OVER (ORDER BY yr))
        / NULLIF(LAG(total_revenue) OVER (ORDER BY yr), 0)
        * 100
    )::NUMERIC, 2) AS yoy_growth_pct
FROM yearly
ORDER BY 1;
```

**********************************************************************

# 🎯 FINAL SQL COMPLETION CHECKLIST


Your SQL analysis is COMPLETE when you have:                         

* Aggregation skills
* Window function mastery
* Business intelligence thinking
* Profit diagnostics ability
* Customer segmentation capability
* Retention analysis knowledge

✔ Executive KPI summary.                                    
✔ Monthly trends + MoM growth.                                               
✔ Category performance + margin classification.                           
✔ State performance .                                
✔ Customer ranking.                                           
✔ Pareto cumulative %.                                             
✔ RFM base metrics                                           
✔ Cohort retention.                                  
✔ Profitability deep dive                                       
 
---